In [0]:
# ========================================
# EXPLORATORY NOTEBOOK
# DATASET: gold_mart_sales_performance
# ========================================

# ========================================
# 0. CONFIGURATION
# ========================================

CATALOG = "ptfrozenfoods_dev"
SOURCE_SCHEMA = "gold"
TARGET_SCHEMA = "gold"

DOMAIN = "marts"
DATASET = "gold_mart_sales_performance"

STORAGE_ACCOUNT = "stptfrozenfoodsdevwe01"
GOLD_CONTAINER = "gold"

FACT_TABLE_NAME = "fact_sales"

FACT_TABLE = f"{CATALOG}.{SOURCE_SCHEMA}.{FACT_TABLE_NAME}"

CANDIDATE_TARGET_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.mart_sales_performance"

GOLD_BASE_PATH = f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

FACT_SOURCE_PATH = f"{GOLD_BASE_PATH}facts/fact_sales/"
CANDIDATE_TARGET_PATH = f"{GOLD_BASE_PATH}marts/mart_sales_performance/"

In [0]:
# ========================================
# 1. CONTEXT SETUP
# ========================================

print("=" * 80)
print("STARTING GOLD EXPLORATORY: gold_mart_sales_performance")
print("=" * 80)

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SOURCE_SCHEMA}")

print("[INFO] Context setup completed successfully.")

STARTING GOLD EXPLORATORY: gold_mart_sales_performance
[INFO] Context setup completed successfully.


In [0]:
# ========================================
# 2. CONFIGURATION SUMMARY
# ========================================

print("=" * 80)
print("GOLD EXPLORATORY NOTEBOOK CONFIGURATION")
print("=" * 80)
print(f"Catalog:                         {CATALOG}")
print(f"Source schema:                   {SOURCE_SCHEMA}")
print(f"Target schema:                   {TARGET_SCHEMA}")
print(f"Domain:                          {DOMAIN}")
print(f"Dataset:                         {DATASET}")
print(f"Fact table:                      {FACT_TABLE}")
print(f"Candidate target table:          {CANDIDATE_TARGET_TABLE}")
print(f"Fact source path:                {FACT_SOURCE_PATH}")
print(f"Candidate target path:           {CANDIDATE_TARGET_PATH}")
print("=" * 80)

GOLD EXPLORATORY NOTEBOOK CONFIGURATION
Catalog:                         ptfrozenfoods_dev
Source schema:                   gold
Target schema:                   gold
Domain:                          marts
Dataset:                         gold_mart_sales_performance
Fact table:                      ptfrozenfoods_dev.gold.fact_sales
Candidate target table:          ptfrozenfoods_dev.gold.mart_sales_performance
Fact source path:                abfss://gold@stptfrozenfoodsdevwe01.dfs.core.windows.net/facts/fact_sales/
Candidate target path:           abfss://gold@stptfrozenfoodsdevwe01.dfs.core.windows.net/marts/mart_sales_performance/


In [0]:
# ========================================
# 3. PRE-CHECKS
# ========================================

print("[INFO] Checking source table availability...")
spark.sql(f"DESCRIBE TABLE {FACT_TABLE}")

print("[INFO] Checking source path access...")
dbutils.fs.ls(FACT_SOURCE_PATH)

print("[INFO] Checking target container access...")
dbutils.fs.ls(f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/")

print("[INFO] Pre-checks completed successfully.")

[INFO] Checking source table availability...
[INFO] Checking source path access...
[INFO] Checking target container access...
[INFO] Pre-checks completed successfully.


In [0]:
# ========================================
# 4. READ SOURCE DATA
# ========================================

print("[INFO] Loading Gold source table...")

df_fact_sales = spark.table(FACT_TABLE)

print("[INFO] Source table loaded successfully.")

print("=" * 80)
print("SOURCE DATA ROW COUNTS")
print("=" * 80)
print(f"Fact sales:          {df_fact_sales.count():,}")
print("=" * 80)

[INFO] Loading Gold source table...
[INFO] Source table loaded successfully.
SOURCE DATA ROW COUNTS
Fact sales:          244,357


In [0]:
# ========================================
# DATASET PREVIEW — INITIAL EXPLORATION
# ========================================

print("=" * 80)
print("DATASET PREVIEW — INITIAL EXPLORATION")
print("=" * 80)

print("\n[INFO] Preview: FACT SALES")
print("-" * 80)
display(df_fact_sales.limit(5))

print("\n[INFO] Printing schema...")
df_fact_sales.printSchema()

print("\n" + "=" * 80)
print("[INFO] Dataset preview completed successfully.")
print("=" * 80)

DATASET PREVIEW — INITIAL EXPLORATION

[INFO] Preview: FACT SALES
--------------------------------------------------------------------------------


item_pedido_id,pedido_id,produto_id,cliente_id,canal_id,fornecedor_id,data_pedido,calendar_year,calendar_quarter,calendar_month,calendar_day,calendar_month_name,calendar_day_of_week,calendar_day_of_week_name,calendar_is_weekend,calendar_is_month_start,calendar_is_month_end,vendedor_id,cidade_entrega,estado_pedido,prazo_entrega_dias,tipo_cliente,cliente_cidade,distrito,segmento,potencial_valor,cluster_comercial,status_cliente,produto_nome,categoria,marca,status_produto,nome_canal,descricao_canal,quantity_sold,preco_lista_unitario,desconto_unitario,preco_venda_unitario,custo_unitario,gross_sales_amount,net_sales_amount,total_cost_amount,gross_margin_amount,line_count,flag_promocao,lote_fornecedor,item_load_date,item_ingestion_timestamp,item_source_file,order_load_date,order_ingestion_timestamp,order_source_file,average_order_value
IT000000027,PED2021011100012,P012,C0136,CH04,F006,2021-01-11,2021,1,1,11,Janeiro,1,Segunda-feira,0,0,0,UNKNOWN,Espinho,Entregue,1,Restaurante,Espinho,Aveiro,Pequeno,Alto,Cluster C,Ativo,Hambúrguer bovino 800g,Carne,Chef Express,Ativo,Marketplace,Plataformas externas de venda,2,5.49,0.0,5.49,3.14,10.98,10.98,6.28,4.7,1,0,L66962,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_orders/load_date=2026-03-17/erp_orders.csv,29.42
IT000000026,PED2021011100012,P020,C0136,CH04,F002,2021-01-11,2021,1,1,11,Janeiro,1,Segunda-feira,0,0,0,UNKNOWN,Espinho,Entregue,1,Restaurante,Espinho,Aveiro,Pequeno,Alto,Cluster C,Ativo,Batata smile 1500g,Batatas,FrioMax,Ativo,Marketplace,Plataformas externas de venda,4,4.61,0.0,4.61,2.34,18.44,18.44,9.36,9.080000000000002,1,0,L39662,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_orders/load_date=2026-03-17/erp_orders.csv,29.42
IT000000028,PED2021011100013,P004,C0136,CH02,F001,2021-01-11,2021,1,1,11,Janeiro,1,Segunda-feira,0,0,0,VEND001,Espinho,Entregue,1,Restaurante,Espinho,Aveiro,Pequeno,Alto,Cluster C,Ativo,Bacalhau desfiado 800g,Peixe,PT Frozen Foods,Ativo,Vendas Externas,Equipa comercial visitando clientes,2,8.72,0.0,8.72,4.9,17.44,17.44,9.8,7.640000000000001,1,0,L28888,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_orders/load_date=2026-03-17/erp_orders.csv,53.79
IT000000030,PED2021011100013,P023,C0136,CH02,F005,2021-01-11,2021,1,1,11,Janeiro,1,Segunda-feira,0,0,0,VEND001,Espinho,Entregue,1,Restaurante,Espinho,Aveiro,Pequeno,Alto,Cluster C,Ativo,Pão de forma 800g,Padaria,Doce Norte,Ativo,Vendas Externas,Equipa comercial visitando clientes,4,2.97,0.15,2.82,1.36,11.88,11.28,5.44,5.839999999999999,1,1,L45385,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_orders/load_date=2026-03-17/erp_orders.csv,53.79
IT000000029,PED2021011100013,P020,C0136,CH02,F002,2021-01-11,2021,1,1,11,Janeiro,1,Segunda-feira,0,0,0,VEND001,Espinho,Entregue,1,Restaurante,Espinho,Aveiro,Pequeno,Alto,Cluster C,Ativo,Batata smile 1500g,Batatas,FrioMax,Ativo,Vendas Externas,Equipa comercial visitando clientes,3,4.93,0.0,4.93,2.24,14.79,14.79,6.720000000000001,8.069999999999999,1,0,L68711,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.c


[INFO] Printing schema...
root
 |-- item_pedido_id: string (nullable = true)
 |-- pedido_id: string (nullable = true)
 |-- produto_id: string (nullable = true)
 |-- cliente_id: string (nullable = true)
 |-- canal_id: string (nullable = true)
 |-- fornecedor_id: string (nullable = true)
 |-- data_pedido: date (nullable = true)
 |-- calendar_year: integer (nullable = true)
 |-- calendar_quarter: integer (nullable = true)
 |-- calendar_month: integer (nullable = true)
 |-- calendar_day: integer (nullable = true)
 |-- calendar_month_name: string (nullable = true)
 |-- calendar_day_of_week: integer (nullable = true)
 |-- calendar_day_of_week_name: string (nullable = true)
 |-- calendar_is_weekend: integer (nullable = true)
 |-- calendar_is_month_start: integer (nullable = true)
 |-- calendar_is_month_end: integer (nullable = true)
 |-- vendedor_id: string (nullable = true)
 |-- cidade_entrega: string (nullable = true)
 |-- estado_pedido: string (nullable = true)
 |-- prazo_entrega_dias: in

In [0]:
# ========================================
# 5. SOURCE VALIDATION
# ========================================

from pyspark.sql import functions as F

required_fact_columns = [
    "pedido_id",
    "data_pedido",
    "produto_id",
    "canal_id",
    "fornecedor_id",
    "calendar_year",
    "calendar_quarter",
    "calendar_month",
    "calendar_day",
    "calendar_month_name",
    "calendar_day_of_week",
    "calendar_is_weekend",
    "calendar_is_month_start",
    "calendar_is_month_end",
    "produto_nome",
    "categoria",
    "marca",
    "status_produto",
    "nome_canal",
    "descricao_canal",
    "quantity_sold",
    "gross_sales_amount",
    "net_sales_amount",
    "total_cost_amount",
    "gross_margin_amount",
    "line_count",
    "average_order_value"
]

missing_fact_columns = [c for c in required_fact_columns if c not in df_fact_sales.columns]

print(f"[INFO] Missing fact columns: {missing_fact_columns}")

if missing_fact_columns:
    raise Exception("Missing required columns in fact_sales.")

print("[INFO] Source validation completed successfully.")

[INFO] Missing fact columns: []
[INFO] Source validation completed successfully.


In [0]:
# ========================================
# 6. INITIAL DATA QUALITY CHECKS
# ========================================

display(
    df_fact_sales.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("data_pedido").alias("distinct_data_pedido"),
        F.countDistinct("produto_id").alias("distinct_produto_id"),
        F.countDistinct("canal_id").alias("distinct_canal_id"),
        F.sum(F.when(F.col("data_pedido").isNull(), 1).otherwise(0)).alias("null_data_pedido"),
        F.sum(F.when(F.col("produto_id").isNull(), 1).otherwise(0)).alias("null_produto_id"),
        F.sum(F.when(F.col("canal_id").isNull(), 1).otherwise(0)).alias("null_canal_id"),
        F.sum(F.when(F.col("net_sales_amount").isNull(), 1).otherwise(0)).alias("null_net_sales_amount"),
        F.sum(F.when(F.col("gross_margin_amount").isNull(), 1).otherwise(0)).alias("null_gross_margin_amount")
    )
)

total_rows,distinct_data_pedido,distinct_produto_id,distinct_canal_id,null_data_pedido,null_produto_id,null_canal_id,null_net_sales_amount,null_gross_margin_amount
244357,1875,36,4,0,0,0,0,0


In [0]:
# ========================================
# 7. BUSINESS PROFILE CHECKS
# ========================================

print("[INFO] Net sales by sales channel")
display(
    df_fact_sales.groupBy("canal_id", "nome_canal")
    .agg(
        F.round(F.sum("net_sales_amount"), 2).alias("total_net_sales"),
        F.round(F.sum("gross_margin_amount"), 2).alias("total_gross_margin"),
        F.sum("quantity_sold").alias("total_quantity_sold"),
        F.countDistinct("pedido_id").alias("total_orders")
    )
    .orderBy(F.desc("total_net_sales"))
)

print("[INFO] Net sales by product")
display(
    df_fact_sales.groupBy("produto_id", "produto_nome", "categoria", "marca")
    .agg(
        F.round(F.sum("net_sales_amount"), 2).alias("total_net_sales"),
        F.round(F.sum("gross_margin_amount"), 2).alias("total_gross_margin"),
        F.sum("quantity_sold").alias("total_quantity_sold"),
        F.countDistinct("pedido_id").alias("total_orders")
    )
    .orderBy(F.desc("total_net_sales"))
)

print("[INFO] Net sales by date")
display(
    df_fact_sales.groupBy("data_pedido")
    .agg(
        F.round(F.sum("net_sales_amount"), 2).alias("total_net_sales"),
        F.round(F.sum("gross_margin_amount"), 2).alias("total_gross_margin"),
        F.countDistinct("pedido_id").alias("total_orders"),
        F.sum("quantity_sold").alias("total_quantity_sold")
    )
    .orderBy("data_pedido")
)

[INFO] Net sales by sales channel


canal_id,nome_canal,total_net_sales,total_gross_margin,total_quantity_sold,total_orders
CH02,Vendas Externas,2308048.52,905638.1,388761,36931
CH03,Telefone,1340274.2,525699.98,221946,22249
CH01,E-commerce,1250732.56,494887.74,207268,20614
CH04,Marketplace,588283.98,233016.95,96818,9732


[INFO] Net sales by product


produto_id,produto_nome,categoria,marca,total_net_sales,total_gross_margin,total_quantity_sold,total_orders
P004,Bacalhau desfiado 800g,Peixe,PT Frozen Foods,1038249.71,387513.65,121966,25720
P007,Cocktail de marisco 1kg,Marisco,FrioMax,541910.89,171906.95,55859,13691
P015,Lasanha bolonhesa 1kg,Pré-cozinhados,Atlântico Select,500059.9,173225.92,73042,13487
P006,Miolo de mexilhão 1kg,Marisco,FrioMax,401121.48,162017.69,53694,12048
P012,Hambúrguer bovino 800g,Carne,Chef Express,365556.18,151150.02,61862,16958
P020,Batata smile 1500g,Batatas,FrioMax,301340.79,139169.86,63635,16016
P014,Almôndegas de novilho 1kg,Carne,Mesa Pronta,210256.47,79797.32,34664,8243
P005,Miolo de camarão 60/80 800g,Marisco,Norte Mar,205950.54,63058.38,18773,5477
P017,Bacalhau com natas 1200g,Pré-cozinhados,PT Frozen Foods,201954.37,67798.69,22567,5801
P003,Potas em anéis 1kg,Peixe,Norte Mar,173512.32,69772.9,24988,6566


[INFO] Net sales by date


data_pedido,total_net_sales,total_gross_margin,total_orders,total_quantity_sold
2021-01-11,1256.6,486.31,35,192
2021-01-12,1103.31,421.07,39,167
2021-01-13,1786.13,715.15,43,274
2021-01-14,882.65,340.37,29,128
2021-01-15,1863.14,738.14,44,275
2021-01-16,1083.24,419.56,25,164
2021-01-17,992.56,374.99,24,140
2021-01-18,1096.47,429.55,34,163
2021-01-19,1961.54,767.97,46,296
2021-01-20,1515.41,590.22,42,234


In [0]:
# ========================================
# 8. BUILD CANDIDATE MART
# ========================================

df_mart_sales_performance_candidate = (
    df_fact_sales
    .groupBy(
        "data_pedido",
        "produto_id",
        "canal_id",
        "calendar_year",
        "calendar_quarter",
        "calendar_month",
        "calendar_day",
        "calendar_month_name",
        "calendar_day_of_week",
        "calendar_day_of_week_name",
        "calendar_is_weekend",
        "calendar_is_month_start",
        "calendar_is_month_end",
        "produto_nome",
        "categoria",
        "marca",
        "status_produto",
        "fornecedor_id",
        "nome_canal",
        "descricao_canal"
    )
    .agg(
        F.round(F.sum("quantity_sold"), 2).alias("quantity_sold"),
        F.round(F.sum("gross_sales_amount"), 2).alias("gross_sales_amount"),
        F.round(F.sum("net_sales_amount"), 2).alias("net_sales_amount"),
        F.round(F.sum("total_cost_amount"), 2).alias("total_cost_amount"),
        F.round(F.sum("gross_margin_amount"), 2).alias("gross_margin_amount"),
        F.countDistinct("pedido_id").alias("order_count"),
        F.sum("line_count").alias("line_count"),
        F.round(F.avg("average_order_value"), 2).alias("average_order_value")
    )
)

print(f"[INFO] Candidate mart row count: {df_mart_sales_performance_candidate.count():,}")

display(df_mart_sales_performance_candidate.limit(10))

[INFO] Candidate mart row count: 113,010


data_pedido,produto_id,canal_id,calendar_year,calendar_quarter,calendar_month,calendar_day,calendar_month_name,calendar_day_of_week,calendar_day_of_week_name,calendar_is_weekend,calendar_is_month_start,calendar_is_month_end,produto_nome,categoria,marca,status_produto,fornecedor_id,nome_canal,descricao_canal,quantity_sold,gross_sales_amount,net_sales_amount,total_cost_amount,gross_margin_amount,order_count,line_count,average_order_value
2025-09-30,P004,CH04,2025,3,9,30,Setembro,2,Terça-feira,0,0,1,Bacalhau desfiado 800g,Peixe,PT Frozen Foods,Ativo,F001,Marketplace,Plataformas externas de venda,4,36.9,35.98,22.66,13.32,2,2,39.41
2026-01-08,P015,CH02,2026,1,1,8,Janeiro,4,Quinta-feira,0,0,0,Lasanha bolonhesa 1kg,Pré-cozinhados,Atlântico Select,Ativo,F001,Vendas Externas,Equipa comercial visitando clientes,10,77.99,73.12,48.13,24.99,3,3,64.24
2024-09-26,P021,CH02,2024,3,9,26,Setembro,4,Quinta-feira,0,0,1,Pão de água 2400g,Padaria,Padaria Premium,Ativo,F005,Vendas Externas,Equipa comercial visitando clientes,6,25.45,24.23,11.71,12.52,2,2,114.98
2023-05-06,P016,CH03,2023,2,5,6,Maio,6,Sábado,1,0,0,Empadão de carne 1200g,Pré-cozinhados,Atlântico Select,Ativo,F001,Telefone,Pedidos feitos por telefone,6,44.4,40.2,27.06,13.14,2,2,63.52
2024-07-26,P007,CH03,2024,3,7,26,Julho,5,Sexta-feira,0,0,1,Cocktail de marisco 1kg,Marisco,FrioMax,Ativo,F002,Telefone,Pedidos feitos por telefone,8,85.2,82.24,56.8,25.44,2,2,55.0
2024-09-21,P007,CH01,2024,3,9,21,Setembro,6,Sábado,1,0,0,Cocktail de marisco 1kg,Marisco,FrioMax,Ativo,F002,E-commerce,Pedidos realizados pelo site B2B/B2C,3,31.44,28.2,20.48,7.72,2,2,35.32
2024-12-17,P010,CH04,2024,4,12,17,Dezembro,2,Terça-feira,0,0,0,Legumes para sopa 1kg,Hortícolas,Campo Verde,Ativo,F004,Marketplace,Plataformas externas de venda,18,50.82,47.54,24.03,23.51,3,4,137.62
2023-09-14,P033,CH02,2023,3,9,14,Setembro,4,Quinta-feira,0,0,0,Smoothie frutos vermelhos 750g,Bebidas,Chef Express,Ativo,F006,Vendas Externas,Equipa comercial visitando clientes,12,41.7,39.42,22.26,17.16,2,2,58.66
2025-01-08,P024,CH02,2025,1,1,8,Janeiro,3,Quarta-feira,0,0,0,Croissant manteiga 1500g,Pastelaria,Doce Norte,Ativo,F005,Vendas Externas,Equipa comercial visitando clientes,19,156.48,152.64,82.55,70.09,5,6,78.62
2026-01-25,P019,CH01,2026,1,1,25,Janeiro,7,Domingo,1,0,1,Batata rústica 2500g,Batatas,FrioMax,Ativo,F002,E-commerce,Pedidos realizados pelo site B2B/B2C,2,11.16,9.82,5.22,4.6,1,1,9.82


In [0]:
# ========================================
# 9. CANDIDATE MART VALIDATION
# ========================================

display(
    df_mart_sales_performance_candidate.select(
        F.count("*").alias("total_rows"),
        F.countDistinct(
            F.concat_ws("||", F.col("data_pedido"), F.col("produto_id"), F.col("canal_id"))
        ).alias("distinct_grain_keys"),
        F.sum(F.when(F.col("data_pedido").isNull(), 1).otherwise(0)).alias("null_data_pedido"),
        F.sum(F.when(F.col("produto_id").isNull(), 1).otherwise(0)).alias("null_produto_id"),
        F.sum(F.when(F.col("canal_id").isNull(), 1).otherwise(0)).alias("null_canal_id"),
        F.sum(F.when(F.col("net_sales_amount").isNull(), 1).otherwise(0)).alias("null_net_sales_amount"),
        F.sum(F.when(F.col("gross_margin_amount").isNull(), 1).otherwise(0)).alias("null_gross_margin_amount")
    )
)

print("[INFO] Candidate mart validation completed successfully.")

total_rows,distinct_grain_keys,null_data_pedido,null_produto_id,null_canal_id,null_net_sales_amount,null_gross_margin_amount
113010,113010,0,0,0,0,0


[INFO] Candidate mart validation completed successfully.


In [0]:
# ========================================
# 10. METRIC RECONCILIATION CHECK
# ========================================

fact_totals = df_fact_sales.select(
    F.round(F.sum("quantity_sold"), 2).alias("fact_quantity_sold"),
    F.round(F.sum("gross_sales_amount"), 2).alias("fact_gross_sales_amount"),
    F.round(F.sum("net_sales_amount"), 2).alias("fact_net_sales_amount"),
    F.round(F.sum("total_cost_amount"), 2).alias("fact_total_cost_amount"),
    F.round(F.sum("gross_margin_amount"), 2).alias("fact_gross_margin_amount"),
    F.countDistinct("pedido_id").alias("fact_order_count"),
    F.sum("line_count").alias("fact_line_count")
)

candidate_totals = df_mart_sales_performance_candidate.select(
    F.round(F.sum("quantity_sold"), 2).alias("mart_quantity_sold"),
    F.round(F.sum("gross_sales_amount"), 2).alias("mart_gross_sales_amount"),
    F.round(F.sum("net_sales_amount"), 2).alias("mart_net_sales_amount"),
    F.round(F.sum("total_cost_amount"), 2).alias("mart_total_cost_amount"),
    F.round(F.sum("gross_margin_amount"), 2).alias("mart_gross_margin_amount"),
    F.sum("order_count").alias("mart_order_count"),
    F.sum("line_count").alias("mart_line_count")
)

display(fact_totals.crossJoin(candidate_totals))

fact_quantity_sold,fact_gross_sales_amount,fact_net_sales_amount,fact_total_cost_amount,fact_gross_margin_amount,fact_order_count,fact_line_count,mart_quantity_sold,mart_gross_sales_amount,mart_net_sales_amount,mart_total_cost_amount,mart_gross_margin_amount,mart_order_count,mart_line_count
914793,5780627.7,5487339.26,3328096.49,2159242.77,89526,244357,914793,5780627.7,5487339.26,3328096.49,2159242.77,216492,244357


In [0]:
# ========================================
# 11. MISSING VALUES ANALYSIS — MART SALES PERFORMANCE
# ========================================

def analyze_missing_values(df, dataset_name):
    print("=" * 80)
    print(f"MISSING VALUES ANALYSIS — {dataset_name.upper()}")
    print("=" * 80)

    total_rows = df.count()

    missing_values_df = df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])

    missing_values_transposed = (
        missing_values_df
        .select(
            F.array([
                F.struct(
                    F.lit(col_name).alias("column_name"),
                    F.col(col_name).alias("null_count")
                ) for col_name in missing_values_df.columns
            ]).alias("missing_data")
        )
        .select(F.explode("missing_data").alias("data"))
        .select(
            F.col("data.column_name"),
            F.col("data.null_count")
        )
        .withColumn(
            "null_percentage",
            F.round((F.col("null_count") / F.lit(total_rows)) * 100, 4)
        )
        .orderBy(F.col("null_percentage").desc())
    )

    display(missing_values_transposed)

    print(f"[INFO] Total rows analyzed: {total_rows:,}")
    print("[INFO] Missing values analysis completed successfully.")
    print("=" * 80)

analyze_missing_values(df_mart_sales_performance_candidate, "mart_sales_performance")

MISSING VALUES ANALYSIS — MART_SALES_PERFORMANCE


column_name,null_count,null_percentage
nome_canal,0,0.0
calendar_quarter,0,0.0
calendar_year,0,0.0
calendar_month_name,0,0.0
calendar_month,0,0.0
net_sales_amount,0,0.0
descricao_canal,0,0.0
calendar_day,0,0.0
categoria,0,0.0
calendar_day_of_week,0,0.0


[INFO] Total rows analyzed: 113,010
[INFO] Missing values analysis completed successfully.


%md
### Gold Layer Exploratory Conclusion — mart_sales_performance

This exploratory notebook validated the dataset required to build the `mart_sales_performance` table in the Gold layer of the PT Frozen Foods data platform. The objective was to confirm data quality, validate the aggregation grain, and ensure readiness for analytical consumption.

---

### Grain Validation

The dimensional grain for the `mart_sales_performance` table was confirmed as:

- **One record per (`data_pedido`, `produto_id`, `canal_id`)**

Validation results:

- Total rows: 113,010  
- Distinct grain keys: 113,010  
- Null values: 0  
- Duplicate values: 0  

This confirms that the dataset is unique, consistent, and correctly aggregated.

---

### Business Value Assessment

The mart enables multidimensional analysis of sales performance across time, products, and sales channels.

#### High-Value Analytical Metrics

- **Net Sales (`net_sales_amount`)**
  - Core revenue metric for performance tracking.

- **Gross Margin (`gross_margin_amount`)**
  - Supports profitability analysis.

- **Quantity Sold (`quantity_sold`)**
  - Enables volume and demand analysis.

- **Total Cost (`total_cost_amount`)**
  - Provides cost structure visibility.

- **Order Count (`order_count`)**
  - Represents distinct orders within the defined grain.

- **Average Order Value (`average_order_value`)**
  - Supports customer behavior analysis.

#### Supporting Attributes

- Calendar attributes (year, month, weekday, etc.)
- Product attributes (name, category, brand, status)
- Sales channel attributes (channel name and description)
- Supplier reference (`fornecedor_id`)

---

### Data Quality Assessment

The dataset demonstrated a high level of quality and consistency.

| Metric | Result |
|--------|--------|
| Total Records | 113,010 |
| Duplicate Records | 0 |
| Null Values | 0 |
| Schema Consistency | Validated |
| Grain Consistency | Confirmed |

---

### Metric Reconciliation

All additive metrics were successfully reconciled with the `fact_sales` table:

| Metric | Status |
|--------|--------|
| quantity_sold | Reconciled |
| gross_sales_amount | Reconciled |
| net_sales_amount | Reconciled |
| total_cost_amount | Reconciled |
| gross_margin_amount | Reconciled |
| line_count | Reconciled |

#### Order Count Behavior

- Distinct orders in fact table: 89,526  
- Aggregated order_count in mart: 216,492  

This difference is expected due to the **non-additive nature** of `order_count`.

The metric represents distinct orders within each aggregation group and should not be summed across dimensions. Global order counts must be calculated using `COUNT(DISTINCT pedido_id)` from the fact table.

---

### Dimensional Model Alignment

The exploratory analysis confirmed alignment with the dimensional model defined in the project documentation.

Validated entities:

- `fact_sales`
- `mart_sales_performance`

The mart structure follows a star schema approach, optimized for analytical workloads.

---

### Medallion Architecture Alignment

| Layer | Status |
|-------|--------|
| Bronze | Completed |
| Silver | Completed |
| Gold | Validated |

---

### Governance and Technology Alignment

| Component | Technology |
|-----------|------------|
| Storage | ADLS Gen2 |
| Format | Delta Lake |
| Processing | Azure Databricks |
| Governance | Unity Catalog |
| Orchestration | Azure Data Factory |

---

### Key Decisions

| Decision | Outcome |
|----------|---------|
| Data source | Directly from `fact_sales` |
| Aggregation grain | `data_pedido`, `produto_id`, `canal_id` |
| Joins with dimensions | Removed (not required) |
| Data quality | Fully validated |
| Metric reconciliation | Confirmed |
| Order count behavior | Documented as non-additive |
| Gold layer readiness | Confirmed |
| Performance optimization | Improved by reducing unnecessary joins |

---

### Conclusion

The dataset has been validated and approved for Gold layer processing. The analysis ensures:

- data quality and integrity;  
- correct aggregation and grain consistency;  
- adherence to dimensional modeling best practices;  
- optimized performance by removing unnecessary transformations;  
- readiness for business intelligence and analytical workloads.

The platform is now ready for the implementation of the `mart_sales_performance` Gold processing notebook.